[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C18_Computer_Vision_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境检查与真实图像管线

确认 numpy 可用，并演示本课的**真实数据加载 + 离线合成回退**管线（optdigits 手写数字 / Grace Hopper 灰度图）。每步都有 `assert` 兜底。

> 本模块无练习；跑通后从**模块 01**开始。

## 1 · numpy 环境自检

In [ ]:
import numpy as np
print('numpy', np.__version__)
rng = np.random.default_rng(0)
x = rng.standard_normal((4, 4))
assert x.shape == (4, 4)
# 本课大量用 sliding_window_view 做卷积/滑窗
from numpy.lib.stride_tricks import sliding_window_view
w = sliding_window_view(np.arange(9).reshape(3, 3), (2, 2))
assert w.shape == (2, 2, 2, 2)
print('✅ numpy 与 sliding_window_view 可用')

## 2 · 真实数据：optdigits（失败回退合成）

用 sklearn 自带的 8×8 手写数字（即 UCI optdigits 的副本）。若 sklearn 不可用或无网络，回退到逻辑等价的合成数字图。下游模块都靠这套接口取数据。

In [ ]:
def load_digits_or_synth(n=200, seed=0):
    '''返回 (images[n,8,8] in [0,1], labels[n]).
       先试真实 optdigits(sklearn)，失败回退合成。'''
    try:
        from sklearn.datasets import load_digits
        d = load_digits()
        X = d.images[:n].astype(float) / 16.0      # 原始 0..16
        y = d.target[:n].astype(int)
        src = 'real optdigits (sklearn)'
    except Exception as e:
        rng = np.random.default_rng(seed)
        X = np.zeros((n, 8, 8)); y = rng.integers(0, 10, size=n)
        for i in range(n):                          # 合成：每类一个带结构的斑块
            cy, cx = rng.integers(2, 6, size=2)
            X[i, max(0,cy-1):cy+2, max(0,cx-1):cx+2] = 1.0
            X[i] += 0.1 * rng.standard_normal((8, 8))
        X = np.clip(X, 0, 1); src = 'synthetic fallback'
    return X, y, src

X, y, src = load_digits_or_synth(200)
print(f'data source: {src}')
print('images', X.shape, 'labels', y.shape, 'value range [%.2f, %.2f]' % (X.min(), X.max()))
assert X.shape == (200, 8, 8) and y.shape == (200,)
assert 0.0 <= X.min() and X.max() <= 1.0
print('✅ 数字数据就绪（真实或合成，接口一致）')

## 3 · 真实图像：Grace Hopper 灰度图（失败回退合成）

经典测试照片，用于模块 01 的滤波/边缘。无网络时合成一张带边缘+渐变+纹理的灰度图，保证边缘/梯度算法有东西可检测。

In [ ]:
def load_gray_photo_or_synth(size=64, seed=0):
    '''返回 (H,W) 灰度图 in [0,1]。先试真实照片，失败回退结构化合成。'''
    try:
        import matplotlib.cbook as cbook, matplotlib.image as mpimg
        with cbook.get_sample_data('grace_hopper.jpg') as f:
            img = mpimg.imread(f).astype(float) / 255.0
        gray = img @ np.array([0.299, 0.587, 0.114])    # RGB->灰度
        # 简单中心裁剪到方形再降采样到 size
        h, w = gray.shape; s = min(h, w)
        gray = gray[(h-s)//2:(h-s)//2+s, (w-s)//2:(w-s)//2+s]
        idx = (np.linspace(0, s-1, size)).astype(int)
        gray = gray[np.ix_(idx, idx)]
        src = 'real grace_hopper.jpg (matplotlib)'
    except Exception:
        rng = np.random.default_rng(seed)
        yy, xx = np.mgrid[0:size, 0:size] / size
        gray = 0.3 + 0.4 * xx                          # 水平渐变
        gray[size//4:3*size//4, size//4:3*size//4] = 0.9  # 一个亮方块(强边缘)
        gray += 0.05 * rng.standard_normal((size, size))  # 纹理/噪声
        gray = np.clip(gray, 0, 1); src = 'synthetic fallback'
    return gray, src

img, src = load_gray_photo_or_synth(64)
print(f'image source: {src}')
print('image', img.shape, 'range [%.2f, %.2f]' % (img.min(), img.max()))
assert img.shape == (64, 64) and 0.0 <= img.min() and img.max() <= 1.0
# 图里应当存在强度变化（否则没有边缘可检测）
assert img.std() > 0.05, '图像应有足够的强度变化'
print('✅ 灰度图就绪（真实或合成），存在可检测的边缘')

## 4 · 小结

- numpy + `sliding_window_view` 是本课卷积/滑窗的主力。
- 两个数据加载器（数字 / 灰度图）都遵循 **try 真实 → except 合成** 的统一接口，下游模块直接复用。
- 一切算法都在已知结构的数据上用 `assert` 验证，保证离线也能跑通。

下一站：**模块 01 · 图像滤波与边缘** —— 从卷积与 Sobel 一路写到简化版 Canny。